# Manageable Feedback Evaluator

The Manageable Feedback Evaluator assesses whether a piece of feedback is focused enough that a student can realistically process and use it. It ensures feedback is clear and prioritizes no more than two issues, so that a student reading feedback knows immediately what to do first. This Evaluator is anchored in the Productive Coaching rubric developed by Quill.org and Leanlab Education, in partnership with Anastasiya A Lipnevich. Given a student response and the teacher feedback on it, the evaluator returns a structured output that includes:

* **manageable_score**: A binary judgment (1 / 0) for whether the feedback is manageable in amount and scope. A 1 applies when the feedback is focused enough that a student could realistically process and act on it in a single revision attempt — generally four sentences or fewer, addressing no more than two clearly prioritized issues — or when the student's response didn't require revision and the feedback appropriately stays brief. A 0 signals feedback that is too dense, too long, or raises too many issues at once for the student to know what to do first, or feedback too thin that leaves out a small fix that would have led to a complete success.
* **reasoning**: A high-level summary of why the feedback received this judgment.
* **key_features**: One entry per factor driving the judgment (length, number of distinct issues raised, whether those issues are clearly prioritized, and whether the student would know immediately what to do first), each with whether it was `met` (1 / 0) and a `justification`.
* **proposed_adjustment**: Suggested moves to bring the feedback into a manageable range (for developers iterating on prompts).

In [1]:
%pip install -qU langchain-openai langchain pydantic textstat typing_extensions

Note: you may need to restart the kernel to use updated packages.


In [2]:
import getpass
import os
from dotenv import load_dotenv
import json
import hashlib
from pathlib import Path
from typing import List, Literal 
from enum import Enum
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
import textstat
from IPython.display import Markdown
import pprint as pp

In [3]:
# Check for the API key
load_dotenv()

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

In [4]:
# -------------------------------------------------------------------------
# Load source-of-truth assets: config.json + every prompt file declared
# in config.steps[*].prompt.messages
# -------------------------------------------------------------------------
# The canonical evaluator definition lives in evals/prompts/purpose.
# All consumers (DS notebook, Python SDK, TypeScript SDK) read these same files,
# so this loader is the pattern the SDK engineer will reproduce.

ASSETS_DIR = Path("./prompts/")

config_path = ASSETS_DIR / "config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

# Load standalone schema files (config.json references them via $ref by path).
with open(ASSETS_DIR / "input_schema.json") as f:
    INPUT_SCHEMA = json.load(f)
with open(ASSETS_DIR / "output_schema.json") as f:
    OUTPUT_SCHEMA = json.load(f)

# Load every prompt message declared in config (system, user, ...). Each
# message has {role, source_path, sha256}. We verify each file's sha256
# matches the declared hash -- drift tripwire #1, applied to every prompt
# regardless of role. CI should promote a mismatch to a hard failure.
PROMPT_MESSAGES = []  # list of (role, text) tuples, preserving config order
for msg_spec in CONFIG["steps"][0]["prompt"]["messages"]:
    role = msg_spec["role"]
    path = ASSETS_DIR / msg_spec["source_path"]
    text = path.read_text()
    actual_sha = hashlib.sha256(text.encode("utf-8")).hexdigest()
    declared_sha = msg_spec["sha256"]
    assert actual_sha == declared_sha, (
        f"prompt drift detected for role={role!r} ({msg_spec['source_path']}): "
        f"declared {declared_sha[:12]}..., actual on disk {actual_sha[:12]}..."
    )
    PROMPT_MESSAGES.append((role, text))

print(
    f"Loaded {CONFIG['evaluator']['id']} "
    f"from {ASSETS_DIR.resolve()}"
)
print(f"  model:       {CONFIG['steps'][0]['model']['name']}")
print(f"  temperature: {CONFIG['steps'][0]['generation']['temperature']}")
print(f"  prompts:")
for msg_spec, (role, text) in zip(CONFIG["steps"][0]["prompt"]["messages"], PROMPT_MESSAGES):
    sha = hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]
    print(f"    {role:>6}  {msg_spec['source_path']:<14} ({len(text):>5} chars, sha {sha})")

Loaded is_manageable from /Users/achi/Documents/evaluators/evals/feedback/writing-feedback/manageable/prompts
  model:       gpt-5.4
  temperature: 1
  prompts:
    system  system.txt     ( 2306 chars, sha 91f7840d9fc0)
     human  user.txt       (   69 chars, sha eaac7e6eadcc)


In [5]:
class ManageableOutput(BaseModel):

    class FeatureJudgment(BaseModel):
            feature: Literal[
                "length",
                "number_of_distinct_issues",
                "clear_priority",
                "student_knows_next_step",
            ] = Field(description="Which key feature this judgment is for.")
            met: Literal[0, 1] = Field(
                description="1 if this feature is satisfied, 0 if not — tied to the specific student response and feedback."
            )
            justification: str = Field(
                description="Short explanation of why this feature is or isn't met, tied to the specific response and feedback."
            )

    reasoning: str = Field(
        description="Step-by-step reasoning tied to the specific student response and feedback."
    )
    key_features: List[FeatureJudgment] = Field(
        description="""One entry per key feature (length, number_of_distinct_issues, clear_priority, student_knows_next_step). Each entry has `feature`, `met`
  (1 or 0), and `justification` (tied to the specific response and feedback)."""
    )
    proposed_adjustment: str = Field(
        description="How the feedback could be changed to better meet the criterion, or a brief note that it already meets the criterion."
    )
    manageable_score: Literal[0, 1] = Field(
        description="""A binary judgment (1 / 0) for whether the feedback is manageable in amount and scope. A 1 applies when the feedback is focused enough that a student could realistically process and act on it in a single revision attempt — generally four sentences or fewer, addressing no more than two clearly prioritized issues — or when the student's response didn't require revision and the feedback appropriately stays brief. A 0 signals feedback that is too dense, too long, or raises too many issues at once for the student to know what to do first, or feedback too thin that leaves out a small fix that would have led to a complete success."""
    )

prompt_vars = {
    "inputVars": ["student_text", "feedback_text"],
    "outputParser": JsonOutputParser(pydantic_object=ManageableOutput),
}

# Sanity: schema fields match config
_defs = CONFIG["output_schema"].get("$defs", {})
def _check(name, cls, schema_props):
    py = set(cls.model_fields.keys())
    sc = set(schema_props.keys())
    assert py == sc, f"{name}: pydantic={py} vs config schema={sc}"
_check("ManageableOutput", ManageableOutput, CONFIG["output_schema"]["properties"])
print("Schema fields match CONFIG['output_schema'].")

Schema fields match CONFIG['output_schema'].


In [6]:
_STEP = CONFIG["steps"][0]

def is_manageable(student_text: str, feedback_text: str):
    """
    Evaluate whether feedback is manageable using the canonical
    config in evals/prompts/purpose/config.json + system.txt + user.txt.

    Returns a dict with full I/O trace fields:
      - rendered_prompt:  the actual list of messages sent to the model
                          (input-side trace).
      - raw_output:       the AIMessage object returned by the LLM
                          (preserves response_metadata, usage_metadata).
      - raw_text:         just the string content of the AIMessage.
      - formatted_output: the parsed dict matching OUTPUT_SCHEMA.
      - usage:            token-usage metadata if the provider returned it.

    The LLM is invoked ONCE; include_raw=True returns both the raw AIMessage
    and the parsed output without a second call.
    """
    # 1. Structured output -- parser.kind == "structured_output" uses the model's
    #    native output enforcement. OUTPUT_SCHEMA is loaded from output_schema.json,
    #    the standalone source of truth. include_raw=True preserves the AIMessage
    #    for tracing alongside the parsed result.

    llm = ChatOpenAI(
        model=_STEP["model"]["name"],
        temperature=_STEP["generation"]["temperature"],
    )
    structured_llm = llm.with_structured_output(OUTPUT_SCHEMA, include_raw=True)

    # 2. Prompt template -- every message's content was loaded from disk
    #    and verified against config in the loader cell. We just feed the
    #    (role, text) tuples straight into ChatPromptTemplate.
    prompt_template = ChatPromptTemplate.from_messages(PROMPT_MESSAGES)

    try:
        inputs = {"student_text": student_text, "feedback_text": feedback_text}

        # Step A: Render the prompt up-front so we can return exactly what
        #         was sent to the model (input-side trace).
        rendered_messages = prompt_template.format_messages(**inputs)

        # Step B: Single LLM call -> raw AIMessage + parsed output dict.
        #         No second LLM call.
        raw = structured_llm.invoke(rendered_messages)

        if raw.get("parsing_error"):
            raise ValueError(f"structured output parsing failed: {raw['parsing_error']}")

        # Step C: Return the full trace dict.
        return {
            "rendered_prompt": [m.model_dump() for m in rendered_messages],
            "raw_output":       raw["raw"],
            "raw_text":         raw["raw"].content,
            "formatted_output": raw["parsed"],
            "usage":            getattr(raw["raw"], "usage_metadata", None),
        }
    except Exception as e:
        return f"Error evaluating text: {e}"

In [7]:
sample_student_text = """Some people think AI-powered pets are a good alternative to real pets because they could help around the house etc."""
sample_feedback_text = """You're right, the AI pets could help around the house. Can you find some other details from the article that you could add to make your claim stronger?"""

case_input = {"student_text": sample_student_text, "feedback_text": sample_feedback_text}
case_output = is_manageable(case_input['student_text'], case_input['feedback_text'])

case_output

{'rendered_prompt': [{'content': 'You are an expert in evaluating written feedback to students, focusing on qualities of\nfeedback that support productive coaching. The ultimate goal of productive coaching is to\nhelp students enter a state of productive struggle — the cognitively challenging but\nemotionally manageable process through which learners develop understanding by grappling\nwith problems just beyond their current capability.\n\nYou are evaluating how well the teacher feedback achieves a specific criterion of\nproductive coaching.\n\nCriterion of productive coaching: Is Manageable\n\nDefinition: Is the amount of feedback manageable for the student?\n\nMeets criterion:\nThe feedback is manageable in amount and scope for the learner. It is focused enough that the student could realistically process and use it. Feedback addresses no more than two clearly prioritized issues. A student reading this would know immediately what to do first. In general, four sentences or fewer will 

In [8]:
import pprint as pp
# I/O trace breakdown -- this is what the SDK engineer will replicate in TS.
print("=" * 60)
print("RENDERED PROMPT (input sent to the LLM)")
print("=" * 60)
pp.pprint(case_output["rendered_prompt"])

print("\n" + "=" * 60)
print("RAW LLM TEXT (model's verbatim output)")
print("=" * 60)
print(case_output["raw_text"])

print("\n" + "=" * 60)
print("PARSED OUTPUT (output_schema)")
print("=" * 60)
pp.pprint(case_output["formatted_output"])

print("\n" + "=" * 60)
print("USAGE METADATA")
print("=" * 60)
pp.pprint(case_output["usage"])

RENDERED PROMPT (input sent to the LLM)
[{'additional_kwargs': {},
  'content': 'You are an expert in evaluating written feedback to students, '
             'focusing on qualities of\n'
             'feedback that support productive coaching. The ultimate goal of '
             'productive coaching is to\n'
             'help students enter a state of productive struggle — the '
             'cognitively challenging but\n'
             'emotionally manageable process through which learners develop '
             'understanding by grappling\n'
             'with problems just beyond their current capability.\n'
             '\n'
             'You are evaluating how well the teacher feedback achieves a '
             'specific criterion of\n'
             'productive coaching.\n'
             '\n'
             'Criterion of productive coaching: Is Manageable\n'
             '\n'
             'Definition: Is the amount of feedback manageable for the '
             'student?\n'
          

In [9]:
fixtures_path = ASSETS_DIR / CONFIG["fixtures"]["sniff_test_path"]
if not fixtures_path.exists():
    print(f"(no fixtures.json yet at {fixtures_path}; skipping fixture run)")
else:
    fixtures = json.loads(fixtures_path.read_text())
    print(f"Loaded {len(fixtures)} fixtures from {fixtures_path.name}\n")

    def _score(predicted, expected):
        if predicted == expected:
            return "exact", 0
        return "fail", None

    results = []
    for fx in fixtures:
        expected = fx["expected"]["manageable_score"]
        out = is_manageable(student_text=fx["input"]["student_text"],
                                       feedback_text=fx["input"]["feedback_text"])
        if isinstance(out, str):
            results.append({"id": fx["id"], "status": "error",
                             "predicted": None, "expected": expected, "error": out})
            continue
        predicted = out["formatted_output"]["manageable_score"]
        status, _ = _score(predicted, expected)
        results.append({"id": fx["id"], "status": status,
                         "predicted": predicted, "expected": expected,
                         "description": fx.get("description", "")})

    print("=" * 78)
    print(f"{'ID':>5}  {'STATUS':<8}  {'PREDICTED':<22}  {'EXPECTED':<22}  DESCRIPTION")
    print("=" * 78)
    for r in results:
        icon = {"exact": "PASS", "adjacent": "PASS*", "fail": "FAIL", "error": "ERR"}[r["status"]]
        print(f"{r['id']:>5}  {icon:<8}  {(r['predicted'] or '-'):<22}  "
              f"{r['expected']:<22}  {r.get('description','')[:25]}")

    n = len(results)
    n_exact = sum(1 for r in results if r["status"] == "exact")
    n_adj = sum(1 for r in results if r["status"] == "adjacent")
    n_fail = sum(1 for r in results if r["status"] == "fail")
    n_err = sum(1 for r in results if r["status"] == "error")
    print("=" * 78)
    print(f"Summary: {n_exact} exact, {n_adj} adjacent, {n_fail} fail, {n_err} error  "
          f"--  total {n}")

Loaded 2 fixtures from fixtures.json

   ID  STATUS    PREDICTED               EXPECTED                DESCRIPTION
    0  PASS      1                       1                       Example 1: Student writes
    1  PASS      1                       1                       Example 2: Student writes
Summary: 2 exact, 0 adjacent, 0 fail, 0 error  --  total 2
